# SafePrice Simulator
## Modeling

In [2]:
import pandas as pd
import numpy as np
import os
import mlflow
import optuna
import joblib
import xgboost as xgb
from interpret.glassbox import ExplainableBoostingRegressor
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

### 1) Preparation Data for Modeling

In [3]:
root_path = os.path.dirname(os.getcwd())

cleaned_datesets_path = os.path.join(root_path, 'datasets/cleaned')

retail_df = pd.read_csv(os.path.join(cleaned_datesets_path, "final_cleaned_retail_dataset.csv"))

retail_df.head()

,InvoiceDate,Quantity,Price,Retail Price Index,is_holiday
0,2010-01-04,3,0.85,859.6,0
1,2010-01-04,4,0.85,859.6,0
2,2010-01-04,3,3.25,859.6,0
3,2010-01-04,36,0.42,859.6,0
4,2010-01-04,2,4.25,859.6,0


In [3]:
print(retail_df.shape)
print(retail_df.dtypes)

(443918, 5)
InvoiceDate            object
Quantity                int64
Price                 float64
Retail Price Index    float64
is_holiday              int64
dtype: object


In [4]:
retail_df['InvoiceDate'] = pd.to_datetime(retail_df ['InvoiceDate'])

retail_df['month'] = retail_df['InvoiceDate'].dt.month
retail_df["day_of_week"] = retail_df['InvoiceDate'].dt.day_of_week

In [5]:
retail_df

,InvoiceDate,Quantity,Price,Retail Price Index,is_holiday,month,day_of_week
0,2010-01-04,3,0.85,859.6,0,1,0
1,2010-01-04,4,0.85,859.6,0,1,0
2,2010-01-04,3,3.25,859.6,0,1,0
3,2010-01-04,36,0.42,859.6,0,1,0
4,2010-01-04,2,4.25,859.6,0,1,0
...,...,...,...,...,...,...,...
443913,2011-12-09,3,1.63,944.4,0,12,4
443914,2011-12-09,1,2.49,944.4,0,12,4
443915,2011-12-09,5,2.46,944.4,0,12,4
443916,2011-12-09,3,2.49,944.4,0,12,4


#### Train Test Split

In [5]:
cutoff_date = pd.to_datetime('2011-05-31')

train_df = retail_df[retail_df['InvoiceDate'] <= cutoff_date]
test_df = retail_df[retail_df['InvoiceDate'] > cutoff_date]

In [6]:
featues = ['Price', 'Retail Price Index', 'is_holiday', 'month', 'day_of_week']
target = ['Quantity']

X_train = train_df[featues]
y_train = train_df[target]
X_test = test_df[featues]
y_test = test_df[target]

### Modeling

#### XGBoost Model

In [8]:
xgb_model = xgb.XGBRFRegressor(
    n_estimators=100, 
    learning_rate=0.1, 
    random_state=1111
)

xgb_model.fit(X_train, y_train)

xgb_pred = xgb_model.predict(X_test)

xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_pred))
xgb_mae = mean_absolute_error(y_test, xgb_pred)

print(f"XGBoost RMSE: {xgb_rmse:.2f}")
print(f"XGBoost MAE: {xgb_mae:.2f}")

XGBoost RMSE: 13.77
XGBoost MAE: 9.33


#### LSTM

In [8]:
scaler_x = MinMaxScaler()
scaler_y = MinMaxScaler()

X_train_scaled = scaler_x.fit_transform(X_train)
X_test_scaled = scaler_x.transform(X_test)
y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1))

X_train_3d = X_train_scaled.reshape((X_train_scaled.shape[0], 1, X_train_scaled.shape[1]))
X_test_3d = X_test_scaled.reshape((X_test_scaled.shape[0], 1, X_test_scaled.shape[1]))

model = Sequential([
    LSTM(32, input_shape=(X_train_3d.shape[1], X_train_3d.shape[2])),
    Dropout(0.1),
    Dense(1)
])

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse')

model.fit(X_train_3d, y_train_scaled, epochs=10, batch_size=64, verbose=0)
scaled_preds = model.predict(X_test_3d)
lstm_preds = scaler_y.inverse_transform(scaled_preds).flatten()

lstm_rmse = np.sqrt(mean_squared_error(y_test, lstm_preds))
lstm_mae = mean_absolute_error(y_test, lstm_preds)

print(f"LSTM RMSE: {lstm_rmse:.2f}")
print(f"LSTM MAE: {lstm_mae:.2f}")

/Users/admin/.local/share/virtualenvs/SafePrice_Simulator-PfLW__QB/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


4433/4433 ━━━━━━━━━━━━━━━━━━━━ 1s 232us/step
LSTM RMSE: 13.91
LSTM MAE: 10.36


#### Explainable Boosting Machine

In [9]:
ebm = ExplainableBoostingRegressor(random_state=1111)

ebm.fit(X_train, y_train)

ebm_pred = ebm.predict(X_test)
ebm_rmse = np.sqrt(mean_squared_error(y_test, ebm_pred))
ebm_mae = mean_absolute_error(y_test, ebm_pred)

print(f"EBM RMSE: {ebm_rmse:.2f}")
print(f"EBM MAE: {ebm_mae:.2f}")

EBM RMSE: 12.83
EBM MAE: 8.70


### Hyperparameter Tuning with MLFLOW

In [ ]:
# mlflow.set_tracking_uri("http://127.0.0.1:8080")
# mlflow.set_experiment("SafePrice_Model_Comparison")

<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1777432563182, experiment_id='1', last_update_time=1777432563182, lifecycle_stage='active', name='SafePrice_Model_Comparison', tags={}, trace_location=None, workspace='default'>

In [21]:
def tune_model(model_name, X_train, y_train, X_test, y_test, n_trials=10):
    y_train_flat = y_train.values.ravel()
    y_test_flat = y_test.values.ravel()

    scaler_x = MinMaxScaler()
    scaler_y = MinMaxScaler()

    X_train_scaled = scaler_x.fit_transform(X_train)
    X_test_scaled = scaler_x.transform(X_test)
    y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1))

    X_train_3d = X_train_scaled.reshape((X_train_scaled.shape[0], 1, X_train_scaled.shape[1]))
    X_test_3d = X_test_scaled.reshape((X_test_scaled.shape[0], 1, X_test_scaled.shape[1]))

    def objective(trial):

        with mlflow.start_run(nested=True, run_name=f"Trial_{trial.number}"):

            if model_name == "XGBoost":
                params = {
                    "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
                    "max_depth": trial.suggest_int("max_depth", 3, 10),
                    "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
                    "subsample": trial.suggest_float("subsample", 0.5, 1.0)
                }
                model = xgb.XGBRegressor(**params, random_state=1111)
                mlflow.log_params(params)
                model.fit(X_train, y_train_flat)
                preds = model.predict(X_test)

            elif model_name == 'EBM':
                params = {
                    "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True),
                    "max_bins": trial.suggest_int("max_bins", 32, 512),
                    "interactions": trial.suggest_int("interactions", 5, 20),
                    "outer_bags": 16
                }
                model = ExplainableBoostingRegressor(**params, random_state=1111)
                mlflow.log_params(params)
                model.fit(X_train, y_train_flat)
                preds = model.predict(X_test)

            
            elif model_name == 'LSTM':
                units = trial.suggest_int("units", 32, 128)
                dropout = trial.suggest_float("dropout", 0.1, 0.4)
                lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)

                model = Sequential([
                    LSTM(units, input_shape=(X_train_3d.shape[1], X_train_3d.shape[2])),
                    Dropout(dropout),
                    Dense(1)
                ])

                model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr), loss='mse')

                model.fit(X_train_3d, y_train_scaled, epochs=10, batch_size=64, verbose=0)

                mlflow.log_params({"units": units, "dropout": dropout, "learning_rate": lr})

                scaled_preds = model.predict(X_test_3d)
                preds = scaler_y.inverse_transform(scaled_preds).flatten()
        
            rmse = np.sqrt(mean_squared_error(y_test_flat, preds))
            mlflow.log_metric("rmse", rmse)

            return rmse
        
    with mlflow.start_run(run_name=f"{model_name}_Optimization"):
        study = optuna.create_study(direction="minimize")
        study.optimize(objective, n_trials=n_trials)

        mlflow.log_params({f"best_{k}": v for k,v in study.best_params.items()})
        mlflow.log_metric("best_rmse", study.best_value)
        print(f"Finished {model_name} tuning. Best RMSE: {study.best_value:.4f}")
        return study.best_params

In [ ]:
# best_xgboost = tune_model("XGBoost", X_train, y_train, X_test, y_test)

[I 2026-04-29 14:51:16,803] A new study created in memory with name: no-name-e72dc694-8f8e-4024-907f-a537ddae9589
[I 2026-04-29 14:51:18,202] Trial 0 finished with value: 13.853448850158047 and parameters: {'n_estimators': 582, 'max_depth': 3, 'learning_rate': 0.20597582586071694, 'subsample': 0.6683455969651411}. Best is trial 0 with value: 13.853448850158047.


🏃 View run Trial_0 at: http://127.0.0.1:8080/#/experiments/1/runs/2f5622d78c714f7eab1c072493b58ebf
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/1


[I 2026-04-29 14:51:19,642] Trial 1 finished with value: 12.927374816772364 and parameters: {'n_estimators': 598, 'max_depth': 4, 'learning_rate': 0.013960425848377235, 'subsample': 0.9358247759094516}. Best is trial 1 with value: 12.927374816772364.


🏃 View run Trial_1 at: http://127.0.0.1:8080/#/experiments/1/runs/952544bc53f6450191a027b91e07e518
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/1


[I 2026-04-29 14:51:22,086] Trial 2 finished with value: 14.378758212868753 and parameters: {'n_estimators': 851, 'max_depth': 5, 'learning_rate': 0.05941463456957863, 'subsample': 0.814343872665092}. Best is trial 1 with value: 12.927374816772364.


...

🏃 View run Trial_7 at: http://127.0.0.1:8080/#/experiments/1/runs/e6467cbe4e2b4725a7ddbd69ae267bcd
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/1


[I 2026-04-29 14:51:39,939] Trial 8 finished with value: 17.61951204176335 and parameters: {'n_estimators': 830, 'max_depth': 10, 'learning_rate': 0.24509928088169228, 'subsample': 0.8052187462359714}. Best is trial 1 with value: 12.927374816772364.


🏃 View run Trial_8 at: http://127.0.0.1:8080/#/experiments/1/runs/64f8bef84e104db0b8259c7b9f708e3f
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/1


[I 2026-04-29 14:51:41,455] Trial 9 finished with value: 13.481186367091652 and parameters: {'n_estimators': 424, 'max_depth': 6, 'learning_rate': 0.016025962130011413, 'subsample': 0.5964441694740513}. Best is trial 1 with value: 12.927374816772364.


🏃 View run Trial_9 at: http://127.0.0.1:8080/#/experiments/1/runs/fce723332a344bbd80e9b9802e232953
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/1
Finished XGBoost tuning. Best RMSE: 12.9274
🏃 View run XGBoost_Optimization at: http://127.0.0.1:8080/#/experiments/1/runs/33b6d3989a2f4b7e9f91422948d159a2
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/1


In [ ]:
# best_ebm = tune_model("EBM", X_train, y_train, X_test, y_test)

[I 2026-04-29 14:51:41,515] A new study created in memory with name: no-name-c8dca468-e094-4b4c-89ea-1746925c82a7
[I 2026-04-29 15:05:42,033] Trial 0 finished with value: 12.935976817480888 and parameters: {'learning_rate': 0.01854213254278948, 'max_bins': 386, 'interactions': 17}. Best is trial 0 with value: 12.935976817480888.


🏃 View run Trial_0 at: http://127.0.0.1:8080/#/experiments/1/runs/eae72879358743e58ca6c81a763c4551
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/1


[I 2026-04-29 15:15:29,055] Trial 1 finished with value: 12.87371562458068 and parameters: {'learning_rate': 0.04455772122882819, 'max_bins': 187, 'interactions': 11}. Best is trial 1 with value: 12.87371562458068.


🏃 View run Trial_1 at: http://127.0.0.1:8080/#/experiments/1/runs/e2a26ec2adac46538520451b69f9cc24
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/1


[I 2026-04-29 15:23:37,267] Trial 2 finished with value: 12.836489193671554 and parameters: {'learning_rate': 0.08719120766887155, 'max_bins': 147, 'interactions': 10}. Best is trial 2 with value: 12.836489193671554.


...

🏃 View run Trial_7 at: http://127.0.0.1:8080/#/experiments/1/runs/44097eb8fb684ea19acfe888c49c8801
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/1


[I 2026-04-29 16:23:07,648] Trial 8 finished with value: 12.729730743253157 and parameters: {'learning_rate': 0.19527421996685604, 'max_bins': 497, 'interactions': 15}. Best is trial 8 with value: 12.729730743253157.


🏃 View run Trial_8 at: http://127.0.0.1:8080/#/experiments/1/runs/06dd650c37524e838236d36955bfb958
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/1


[I 2026-04-29 16:37:42,556] Trial 9 finished with value: 13.11382064755528 and parameters: {'learning_rate': 0.01834000479783029, 'max_bins': 96, 'interactions': 10}. Best is trial 8 with value: 12.729730743253157.


🏃 View run Trial_9 at: http://127.0.0.1:8080/#/experiments/1/runs/327363944f754d34818326924fedae42
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/1
Finished EBM tuning. Best RMSE: 12.7297
🏃 View run EBM_Optimization at: http://127.0.0.1:8080/#/experiments/1/runs/036131eac4e443e5a32a726dbf1fa81a
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/1


In [ ]:
# best_lstm = tune_model("LSTM", X_train, y_train, X_test, y_test)

[I 2026-04-29 17:27:20,900] A new study created in memory with name: no-name-d3230d58-d063-454e-a931-4bc77501a020
/Users/admin/.local/share/virtualenvs/SafePrice_Simulator-PfLW__QB/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


4433/4433 ━━━━━━━━━━━━━━━━━━━━ 1s 293us/step


[I 2026-04-29 17:27:57,279] Trial 0 finished with value: 13.610046580372966 and parameters: {'units': 101, 'dropout': 0.3818924376272414, 'lr': 0.0031878310319358957}. Best is trial 0 with value: 13.610046580372966.


🏃 View run Trial_0 at: http://127.0.0.1:8080/#/experiments/1/runs/76f98bcf2aad46d2bd3608cd4db380c5
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/1


/Users/admin/.local/share/virtualenvs/SafePrice_Simulator-PfLW__QB/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


...

🏃 View run Trial_8 at: http://127.0.0.1:8080/#/experiments/1/runs/1e97b075ac18422bb2c5245531700c67
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/1


/Users/admin/.local/share/virtualenvs/SafePrice_Simulator-PfLW__QB/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


4433/4433 ━━━━━━━━━━━━━━━━━━━━ 1s 268us/step


[I 2026-04-29 17:32:52,511] Trial 9 finished with value: 13.832504492751564 and parameters: {'units': 55, 'dropout': 0.2016016163001344, 'lr': 0.0023158975388261516}. Best is trial 0 with value: 13.610046580372966.


🏃 View run Trial_9 at: http://127.0.0.1:8080/#/experiments/1/runs/2113c39e79074964bdb9264f75043602
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/1
Finished LSTM tuning. Best RMSE: 13.6100
🏃 View run LSTM_Optimization at: http://127.0.0.1:8080/#/experiments/1/runs/5465dedc05bb437592b1cbcee82fc8aa
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/1


#### Best Models Saved

In [ ]:
# models_path = os.path.join(root_path, 'models')
# joblib.dump(best_xgboost, os.path.join(models_path, 'best_xgboost_model.pkl'))
# joblib.dump(best_ebm, os.path.join(models_path, 'best_ebm_model.pkl'))
# joblib.dump(best_lstm, os.path.join(models_path, 'best_lstm_model.pkl'))
# print("Models Successfully Saved.")

Models Successfully Saved.
